# Лабораторная работа №5 по дисциплине "Язык программирования Python".  

# Тема: ООП. Работа с базами данных

## Цель занятия
Закрепить навыки использования ООП в языке Python. Научиться использовать базы данных на примере СУБД sqlite3

**Общее время выполнения: 90 минут**

## Задание 1. Базы данных и  ООП 

**Темы:** объектно-ориентированное программирование, базы данных .


**Что нужно сделать:**
Создайте абстрактный (или просто базовый) класс Animal с атрибутами name и age. Определите в нём метод make_sound() (с заглушкой), который должен быть переопределён в потомках.  
Создайте классы Dog и Cat, наследующие Animal. В конструкторе они принимают дополнительные параметры: breed (порода) для собаки и color (окрас) для кошки.  
Создайте класс AnimalDB, который создаёт таблицу animals с полями: id, name, age, type (dog/cat), breed (может быть NULL), color (может быть NULL). Реализуйте методы:  
- save(animal) – сохраняет любое животное (определяет его тип и заполняет нужные поля).
- load_all_animals() – загружает всех животных из БД и возвращает список объектов типа Dog или Cat.  

**Проверка работы:**  
Создайте экземпляры Dog и Cat, сохраните их в БД с помощью save(). Загрузите всех животных и для каждого вызовите метод make_sound(), который должен выводить соответствующий звук.

**Для решения задачи дополните шаблон кода, который дан ниже.**

Результат работы должен быть представлен в следующем виде:
```
Сохраняем животных в БД:
  - Собака: Рекс, возраст 5, порода Овчарка
  - Собака: Бобик, возраст 3, порода Дворняга
  - Кошка: Мурка, возраст 4, окрас Серая
  - Кошка: Барсик, возраст 2, окрас Рыжий

==================================================
Загружаем всех животных из БД и вызываем их звуки:

Собака: Рекс, возраст 5, порода Овчарка
Звук: Гав-гав!

Собака: Бобик, возраст 3, порода Дворняга
Звук: Гав-гав!

Кошка: Мурка, возраст 4, окрас Серая
Звук: Мяу-мяу!

Кошка: Барсик, возраст 2, окрас Рыжий
Звук: Мяу-мяу!

==================================================
Демонстрация полиморфизма вне БД:
Собака: Рекс, возраст 5, порода Овчарка -> Гав-гав!
Собака: Бобик, возраст 3, порода Дворняга -> Гав-гав!
Кошка: Мурка, возраст 4, окрас Серая -> Мяу-мяу!
Кошка: Барсик, возраст 2, окрас Рыжий -> Мяу-мяу!
```
**Ключевые моменты**  
- Абстрактный класс Animal задаёт интерфейс (метод make_sound())  
- Наследование позволяет переиспользовать код и добавлять специфичные атрибуты  
- Полиморфизм проявляется при вызове make_sound() – объект сам знает, как ему "говорить"  
- Инкапсуляция работы с БД в отдельном классе AnimalDB  

In [16]:
# Ваш код здесь
import sqlite3
from abc import ABC, abstractmethod

# ---------- Базовый и дочерние классы ----------
class Animal(ABC):
    """Абстрактный базовый класс Animal"""
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    @abstractmethod
    def make_sound(self) -> str:
        """Абстрактный метод, который должен быть переопределен в потомках."""
# Вставьте описание класса сюда

class Dog(Animal):
    """Класс Dog, наследующий Animal"""
    def __init__(self, name: str, age: int, breed: str):
        super().__init__(name, age)
        self.breed = breed

    def make_sound(self) -> str:
        return "Гав-гав!"

    def __str__(self):
        return f"Собака: {self.name}, возраст {self.age}, порода {self.breed}"
# Вставьте описание класса сюда

class Cat(Animal):
    """Класс Cat, наследующий Animal"""
    def __init__(self, name: str, age: int, color: str):
        super().__init__(name, age)
        self.color = color

    def make_sound(self) -> str:
        return "Мяу-мяу!"

    def __str__(self):
        return f"Кошка: {self.name}, возраст {self.age}, окрас {self.color}"
# Вставьте описание класса сюда


# ---------- Класс для работы с базой данных ----------
class AnimalDB:
    def __init__(self, db_name="memory"):
        self.conn = sqlite3.connect(":memory:")
        self.cursor = self.conn.cursor()
        self._create_table()
        """Инициализация: создание подключения и таблицы animals"""
    
    # Вставьте описание конструктора сюда
    
    def _create_table(self):
        """Создание таблицы animals для хранения разных типов животных"""
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS animals (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                age INTEGER NOT NULL,
                type TEXT NOT NULL,
                breed TEXT,
                color TEXT
            )
        """)
        self.conn.commit()
    # Вставьте описание таблицы сюда   
    
    def save(self, animal):
        """
        Сохраняет любое животное в базу данных.
        Определяет тип животного и заполняет соответствующие поля.
        """
        if isinstance(animal, Dog):
            animal_type = "dog"
            breed = animal.breed
            color = None
        elif isinstance(animal, Cat):
            animal_type = "cat"
            breed = None
            color = animal.color
        else:
            raise ValueError("Неизвестный тип животного")

        self.cursor.execute(
            """
            INSERT INTO animals (name, age, type, breed, color)
            VALUES (?, ?, ?, ?, ?)
        """,
            (animal.name, animal.age, animal_type, breed, color),
        )
        self.conn.commit()
    # Вставьте описание метода сюда
    
    def load_all_animals(self):
        """
        Загружает всех животных из базы данных и возвращает список объектов animals
        типа Dog или Cat (полиморфное поведение).
        """
       # Вставьте описание метода сюда           
        self.cursor.execute("SELECT name, age, type, breed, color FROM animals")
        rows = self.cursor.fetchall()

        animals = []
        for row in rows:
            name, age, animal_type, breed, color = row
            if animal_type == "dog":
                animals.append(Dog(name, age, breed))
            elif animal_type == "cat":
                animals.append(Cat(name, age, color))

        return animals

    
    def close(self):
        """Закрывает соединение с базой данных"""
        self.conn.close()


# ---------- ПРИМЕР ИСПОЛЬЗОВАНИЯ ----------

# Создаём экземпляры животных
dog1 = Dog("Рекс", 5, "Овчарка")
dog2 = Dog("Бобик", 3, "Дворняга")
cat1 = Cat("Мурка", 4, "Серая")
cat2 = Cat("Барсик", 2, "Рыжий")
    
# Сохраняем животных в базу данных
# Вставьте код для сохранения животных сюда
db = AnimalDB()
db.save(dog1)
db.save(dog2)
db.save(cat1)
db.save(cat2)

print("\n" + "="*50)
print("Загружаем всех животных из БД и вызываем их звуки:")
    
# Загружаем всех животных (полиморфное поведение)
# Вставьте код для загрузки животных и вывода звуков сюда   
loaded_animals = db.load_all_animals()

for animal in loaded_animals:
    print(animal.name, animal.make_sound())
# Закрываем соединение
db.close()
    
print("\n" + "="*50)
print("Демонстрация полиморфизма вне БД:")
# Дополнительная демонстрация полиморфизма

animals_list = [dog1, dog2, cat1, cat2]
for animal in animals_list:
        print(f"{animal} -> {animal.make_sound()}")

# Вставьте код для вывода всего списка животных сюда


Загружаем всех животных из БД и вызываем их звуки:
Рекс Гав-гав!
Бобик Гав-гав!
Мурка Мяу-мяу!
Барсик Мяу-мяу!

Демонстрация полиморфизма вне БД:
Собака: Рекс, возраст 5, порода Овчарка -> Гав-гав!
Собака: Бобик, возраст 3, порода Дворняга -> Гав-гав!
Кошка: Мурка, возраст 4, окрас Серая -> Мяу-мяу!
Кошка: Барсик, возраст 2, окрас Рыжий -> Мяу-мяу!
